In [1]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from utils import write_csv
from utils import display_excel_sheets

from models import (
    AssessmentReport,
    DataSource,
    GasType,
    Emissions,
    EmissionsCO2e,
    EmissionsTotalCO2e,
    EmissionsTotalSector,
    AggregationType,
)


In [3]:

# ------------------------------------------
# get actors
# ------------------------------------------

fl_actors = (
    #Path(__file__).resolve().parent.parent.parent
    Path.cwd().resolve().parent.parent
    / "data"
    / "iso-3166-1"
    / "raw"
    / "countries.csv"
)

df_actors = (
    pd.read_csv(fl_actors, keep_default_na=False)
    .loc[:, ["#country_code_alpha2", "country_code_alpha3"]]
    .rename(columns={"#country_code_alpha2":"alpha2", "country_code_alpha3":"alpha3"})
    )

In [77]:

fl = (
    #Path(__file__).resolve().parent.parent.parent
    Path.cwd().resolve().parent.parent
    / "data"
    / "unfccc-annex1" 
    / "raw"
    / "GBR_ghg_profile.xlsx"
)

files = (
    #Path(__file__).resolve().parent.parent.parent
    Path.cwd().resolve().parent.parent
    / "data"
    / "unfccc-annex1" 
    / "raw"
).glob("*_ghg_profile.xlsx")

In [78]:
# ------------------------------------------
# DataSource table
# ------------------------------------------
# https://unfccc.int/topics/mitigation/resources/registry-and-data/ghg-data-from-unfccc
datasource_name = "ghg-profiles-annex-1"
publisher = "UNFCCC"
#actor_id = df_actors.loc[df_actors["alpha3"] == iso3, "alpha2"].iat[0]

datasource = DataSource(
    id=f"{publisher}:{datasource_name}",
    name="GHG Profiles - Annex I",
    publisher=publisher,
    published_date=datetime.strptime("2025-10-28", "%Y-%m-%d"),
    version="20251028",
    url = "https://di.unfccc.int/ghg_profile_annex1"
    #url=f"https://di.unfccc.int/ghg_profiles/annexOne/{iso3}/{iso3}_ghg_profile.xlsx",
)


In [79]:
fl_codes = Path.cwd().resolve().parent.parent / "data" / "crt-on-nir" / "processed" / "sector.csv"
#Path(__file__).resolve().parent.parent.parent
df_codes = (
    pd.read_csv(fl_codes)
    .rename(columns={"id":"id_sector"})
    .loc[:, ["id_sector", "code", "name"]]
)

In [80]:
from tqdm import tqdm

In [81]:
#===================
# data by sector
#===================
kt_to_tonne = 1000

list_of_dfs = []

for fl in tqdm(files):
    iso3 = Path(fl).stem.split("_")[0]
    actor_id = df_actors.loc[df_actors["alpha3"] == iso3, "alpha2"].iat[0]

    df_tmp = (
        pd.read_excel(fl, sheet_name="Data_by_sector")
        .iloc[
        lambda d: slice(
            d.index[d["GHG emissions, kt CO2 equivalent"].eq('Breakdown by sub-sectors')][0] + 1,
            d.index[d["GHG emissions, kt CO2 equivalent"].eq('Average annual change, in percent')][0]
        )
        ]
        .drop(columns=["Base year"], errors="ignore")
        .drop(columns=["1990.1"], errors="ignore") # this shows in KAZ for some reason
        .melt(id_vars="GHG emissions, kt CO2 equivalent", var_name='year', value_name='emissions_kt_co2_eq')
        .rename(columns={"GHG emissions, kt CO2 equivalent":"sector"})
        .loc[
            lambda d: pd.to_numeric(d["emissions_kt_co2_eq"], errors="coerce").notna()
        ]
        .assign(
            code=lambda d: d["sector"].str.split(" ", n=1).str[0],
            name=lambda d: d["sector"].str.split(" ", n=1).str[1],
            emissions=lambda d: d["emissions_kt_co2_eq"] * kt_to_tonne,
            units = "CO2 * tonne / yr"
        )
        .assign(
            code = lambda d: d['code'].str.rstrip('.'),
            actor_id = actor_id
        )
        .merge(df_codes, on=["code"])
        .loc[:, ["id_sector", "actor_id", "year", "emissions", "units"]]
        .astype({
            "id_sector": str,
            "actor_id": str, 
            "year": int, 
            "emissions": float, 
            "units": str})
    )

    list_of_dfs.append(df_tmp)


df = pd.concat(list_of_dfs).astype({"actor_id": str, "year": int, "emissions": float, "units": str})

43it [00:01, 32.59it/s]


In [75]:
df.head()

,id_sector,actor_id,year,emissions,units
0,crt:1,RO,1990,1.850428e+08,CO2 * tonne / yr
1,crt:1.A.1,RO,1990,7.169467e+07,CO2 * tonne / yr
2,crt:1.A.2,RO,1990,5.410835e+07,CO2 * tonne / yr
3,crt:1.A.3,RO,1990,1.243177e+07,CO2 * tonne / yr
4,crt:1.A.4,RO,1990,1.138659e+07,CO2 * tonne / yr


In [ ]:
# data by gas
kt_to_tonne = 1000

gases = ['CO₂', 'CH₄', 'N₂O', 'Aggregate F-gases']
#df_gas.query("'GHG emissions, kt CO2 equivalent'.isin(@gases)")

subscript_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")


# with LULUCF
df_gas = (
    pd.read_excel(fl, sheet_name="Data_by_gas")
    .iloc[
    lambda d: slice(
        d.index[d["GHG emissions, kt CO2 equivalent"].eq('GHG emissions without LULUCF')][0] + 1,
        d.index[d["GHG emissions, kt CO2 equivalent"].eq('GHG emissions with LULUCF')][0]
    )
    ]
    .loc[lambda d: d["GHG emissions, kt CO2 equivalent"].isin(gases)]
    .drop(columns=["Base year"])
    .melt(id_vars="GHG emissions, kt CO2 equivalent", var_name='year', value_name='emissions_kt_co2_eq')
    .rename(columns={"GHG emissions, kt CO2 equivalent":"gas"})
    .loc[
        lambda d: pd.to_numeric(d["emissions_kt_co2_eq"], errors="coerce").notna()
    ]
    .assign(
        emissions=lambda d: d["emissions_kt_co2_eq"] * kt_to_tonne,
        units = "CO2 * tonne / yr"
    )
    .assign(
        actor_id = actor_id,
        gas = lambda d: d["gas"].str.translate(subscript_map).str.replace("Aggregate F-gases", "FGASES", regex=False)
    )
    #.merge(df_codes, on=["code"])
    .loc[:, ["actor_id", "year", "emissions", "gas", "units"]]
)


#df['sector'].str.split(' ', n=1, expand=True).rename(columns={0:'code', 1:'name'})

In [202]:
df_gas

,actor_id,year,emissions,gas,units
0,GB,1990,603666158.680242,CO2,CO2 * tonne / yr
1,GB,1990,145315661.611159,CH4,CO2 * tonne / yr
2,GB,1990,42524965.408588,N2O,CO2 * tonne / yr
3,GB,1990,14795056.815133,F-gases,CO2 * tonne / yr
4,GB,1991,611180125.323394,CO2,CO2 * tonne / yr
...,...,...,...,...,...
123,GB,2020,12097203.247135,F-gases,CO2 * tonne / yr
124,GB,2021,348409722.355914,CO2,CO2 * tonne / yr
125,GB,2021,51738658.095558,CH4,CO2 * tonne / yr
126,GB,2021,17906347.685962,N2O,CO2 * tonne / yr


In [196]:
ord('0')

48

In [189]:
df_gas["gas"]

0                    CO₂
1                    CH₄
2                    N₂O
3      Aggregate F-gases
4                    CO₂
             ...        
123    Aggregate F-gases
124                  CO₂
125                  CH₄
126                  N₂O
127    Aggregate F-gases
Name: gas, Length: 128, dtype: object

In [184]:
gases = ['CO₂', 'CH₄', 'N₂O', 'Aggregate F-gases']
#df_gas.query("'GHG emissions, kt CO2 equivalent'.isin(@gases)")
df_gas.loc[df_gas["GHG emissions, kt CO2 equivalent"].isin(gases)]



,"GHG emissions, kt CO2 equivalent",Base year,1990,1991,1992,1993,1994,1995,1996,1997,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
1,CO₂,603666.15868,603666.15868,611180.125323,595547.681984,581571.284779,575574.302712,567718.832279,588352.233291,564846.400845,...,489083.109737,479088.606963,440062.648355,423772.572592,400645.322536,388415.420158,380582.436996,366942.115352,327164.209878,348409.722356
2,CH₄,145315.661611,145315.661611,146184.121457,146004.779545,144399.804472,136756.202151,138616.551615,137770.021394,135013.99593,...,65358.153053,60648.042511,58440.24988,57334.286776,55295.467013,55679.355139,55171.663483,54691.285427,52265.807627,51738.658096
3,N₂O,42524.965409,42524.965409,42676.528178,38257.240721,34460.028754,35065.549863,33864.783273,33962.504737,34250.470533,...,18370.352332,18378.059726,18926.31458,18748.578263,18292.899882,18670.548497,18475.030316,18575.744048,17437.855861,17906.347686
4,Aggregate F-gases,14795.056815,14795.056815,15129.249087,15089.818243,15501.681411,16328.058201,17444.049241,18299.966209,20115.073689,...,13516.867715,13925.952639,14000.948726,14024.755585,14075.617154,14108.891072,13640.82052,13024.50144,12097.203247,11434.742961


In [176]:
df_gas.index[df_gas["GHG emissions, kt CO2 equivalent"].eq('GHG emissions without LULUCF')]

Index([0, 26], dtype='int64')

In [ ]:

# load data and define output directory
data_dir = (
    Path(__file__).resolve().parent.parent.parent / "data" / "eccc-ghg-inventory"
)
output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

write_csv(
    output_dir=output_dir,
    name=DataSource.__tablename__,
    data=datasource.model_dump(),
    mode="w",
)

# ------------------------------------------
# get actors
# ------------------------------------------

fl_actors = (
    Path(__file__).resolve().parent.parent.parent
    / "data"
    / "iso-3166-2"
    / "processed"
    / "actor.csv"
)

df_actors = pd.read_csv(fl_actors, keep_default_na=False)
df_actors = df_actors.loc[df_actors["is_part_of"] == "CA", ["id"]].rename(
    columns={"id": "actor_id"}
)

# ------------------------------------------
# get gwps
# ------------------------------------------

fl_gwp = (
    Path(__file__).resolve().parent.parent.parent
    / "data"
    / "gwp"
    / "processed"
    / "gwp.csv"
)

df_gwp = (
    pd.read_csv(fl_gwp)
    .loc[:, ["id", "gwp", "gas", "assessment_report"]]
    .rename(columns={"id": "gwp_id"})
)

# ------------------------------------------
# get sectors
# ------------------------------------------
fl_sector = (
    Path(__file__).resolve().parent.parent.parent
    / "data"
    / "crt-on-nir"
    / "processed"
    / "sector.csv"
)
df_sector_tmp = pd.read_csv(fl_sector)
df_sector = (
    df_sector_tmp.query("code.isin(['0','1','2','3','4','5','6'])")
    .assign(
        name_sanitized=lambda x: x.apply(
            lambda row: row["name"].replace("-", " ").replace(",", "").lower(),
            axis=1,
        )
    )
    .rename(columns={"id": "sector_id"})
)


In [49]:
fl = "/Users/lukegloege/projects/ghg-tracker-database/data/unfccc-annex1/GBR_ghg_profile.xlsx"

In [4]:
display_excel_sheets(fl)

['Data_by_sector', 'Data_by_gas', 'Summary Data', 'ChartData']

In [ ]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


def full_year(two_digit):
    year = int(two_digit)
    if year >= 50:
        return 1900 + year
    else:
        return 2000 + year


def main():
    # ------------------------------------------
    # DataSource table
    # ------------------------------------------
    datasource_name = "official-ghg-inventory"
    publisher = "ECCC"

    datasource = DataSource(
        id=f"{publisher}:{datasource_name}",
        name="Official Greenhouse Gas Inventory",
        publisher=publisher,
        published_date=datetime.strptime("2025-03-21", "%Y-%m-%d"),
        version="20250321",
        url="https://data.ec.gc.ca/data/substances/monitor/canada-s-official-greenhouse-gas-inventory/A-IPCC-Sector/?lang=en",
    )

    # load data and define output directory
    data_dir = (
        Path(__file__).resolve().parent.parent.parent / "data" / "eccc-ghg-inventory"
    )
    output_dir = data_dir / "processed"
    output_dir.mkdir(parents=True, exist_ok=True)

    write_csv(
        output_dir=output_dir,
        name=DataSource.__tablename__,
        data=datasource.model_dump(),
        mode="w",
    )

    # ------------------------------------------
    # get actors
    # ------------------------------------------

    fl_actors = (
        Path(__file__).resolve().parent.parent.parent
        / "data"
        / "iso-3166-2"
        / "processed"
        / "actor.csv"
    )

    df_actors = pd.read_csv(fl_actors, keep_default_na=False)
    df_actors = df_actors.loc[df_actors["is_part_of"] == "CA", ["id"]].rename(
        columns={"id": "actor_id"}
    )

    # ------------------------------------------
    # get gwps
    # ------------------------------------------

    fl_gwp = (
        Path(__file__).resolve().parent.parent.parent
        / "data"
        / "gwp"
        / "processed"
        / "gwp.csv"
    )

    df_gwp = (
        pd.read_csv(fl_gwp)
        .loc[:, ["id", "gwp", "gas", "assessment_report"]]
        .rename(columns={"id": "gwp_id"})
    )

    # ------------------------------------------
    # get sectors
    # ------------------------------------------
    fl_sector = (
        Path(__file__).resolve().parent.parent.parent
        / "data"
        / "crt-on-nir"
        / "processed"
        / "sector.csv"
    )
    df_sector_tmp = pd.read_csv(fl_sector)
    df_sector = (
        df_sector_tmp.query("code.isin(['0','1','2','3','4','5','6'])")
        .assign(
            name_sanitized=lambda x: x.apply(
                lambda row: row["name"].replace("-", " ").replace(",", "").lower(),
                axis=1,
            )
        )
        .rename(columns={"id": "sector_id"})
    )

    # container to hold all the dataframes
    list_of_dfs = []

    files = (data_dir / "raw").glob("EN_GHG_IPCC_??.xlsx")

    for fl in files:
        actor_id = f"CA-{fl.stem[-2:]}"

        actor_is_valid = df_actors["actor_id"].isin([actor_id]).any()

        if not actor_is_valid:
            raise ValueError(f"Actor ID {actor_id} not found in df_actors.")

        sheets = display_excel_sheets(fl)
        sheets_that_are_years = [sheet for sheet in sheets if is_number(sheet)]

        for sheet_name in sheets_that_are_years:
            # sheet_name = "22"
            year = full_year(sheet_name)
            df_raw = pd.read_excel(fl, sheet_name=sheet_name, header=3)

            # get the want to end at
            idx = df_raw.index[df_raw["Unnamed: 0"] == "Notes:"].tolist()[0]

            df_tmp = (
                df_raw.rename(columns={"Unnamed: 0": "sector"})
                .loc[:, ["sector", "CO2", "CH4", "N2O", "HFCsa", "PFCsa", "SF6", "NF3"]]
                .rename(columns={"HFCsa": "HFCs", "PFCsa": "PFCs"})
                .loc[: (idx - 1)]
            )
            sectors = [
                "TOTALb",
                "ENERGY",
                "INDUSTRIAL PROCESSES AND PRODUCT USE",
                "AGRICULTURE",
                "WASTE",
                "LAND USE, LAND-USE CHANGE AND FORESTRY",
            ]
            all_gases = ["CO2", "CH4", "N2O", "HFCs", "PFCs", "SF6", "NF3"]
            F_gases = ["HFCs", "PFCs", "SF6", "NF3"]
            main_three = ["CO2", "CH4", "N2O"]

            # get units and gwps
            df_units = df_tmp[1:2].melt(
                value_vars=all_gases, id_vars=None, value_name="unit", var_name="gas"
            )

            df_data = (
                df_tmp.loc[2:, :]
                .melt(
                    value_vars=all_gases,
                    id_vars="sector",
                    value_name="emissions",
                    var_name="gas",
                )
                .dropna(subset=["emissions"])
                .query("sector.isin(@sectors)")
                .query('emissions != "-"')
            )
            fgases = ["HFCs", "PFCs", "NF3", "SF6"]
            main_gases = ["CO2", "CH4", "N2O"]

            # Make the FGASES totals by sector
            fgases_sum = (
                df_tmp.loc[2:, :]
                .melt(
                    value_vars=all_gases,
                    id_vars="sector",
                    value_name="emissions",
                    var_name="gas",
                )
                .dropna(subset=["emissions"])
                .query("sector.isin(@sectors)")
                .query('emissions != "-"')
                .query("gas in @fgases")
                .groupby("sector", as_index=False)
                .agg({"emissions": "sum"})
                .assign(gas="FGASES", units="kt CO2 / yr")
            )

            # Keep original CO2, CH4, N2O rows
            other_gases = (
                df_tmp.loc[2:, :]
                .melt(
                    value_vars=all_gases,
                    id_vars="sector",
                    value_name="emissions",
                    var_name="gas",
                )
                .dropna(subset=["emissions"])
                .query("sector.isin(@sectors)")
                .query('emissions != "-"')
                .query("gas in @main_gases")
                .assign(
                    units=lambda x: x.apply(
                        lambda row: f"kt {row['gas']} /year", axis=1
                    )
                )
            )

            # Concatenate back together
            df_data = (
                pd.concat([other_gases, fgases_sum], ignore_index=True)
                .sort_values(["sector", "gas"])
                .reset_index(drop=True)
                .rename(columns={"emissions": "emissions_kt"})
                .assign(actor_id=actor_id, year=year)
            )
            kt_to_tonne = 1_000

            df_sheet = (
                df_data.assign(
                    name_sanitized=lambda x: x.apply(
                        lambda row: row["sector"]
                        .replace("-", " ")
                        .replace(",", "")
                        .lower(),
                        axis=1,
                    )
                )
                .merge(
                    df_sector.loc[:, ["sector_id", "code", "name_sanitized"]],
                    on="name_sanitized",
                )
                .assign(
                    datasource_id=datasource.id,
                    assessment_report="AR5",
                    emissions_tonnes=lambda x: x["emissions_kt"] * kt_to_tonne,
                    units=lambda x: x["units"].str.replace("kt", "tonne", regex=False),
                )
                .astype({"emissions_tonnes": float})
                .merge(df_gwp, on=["gas", "assessment_report"], how="left")
                .assign(
                    emissions_tonnes_co2e=lambda x: np.round(
                        x["emissions_tonnes"] * x["gwp"], 11
                    ).fillna(x["emissions_tonnes"]),
                    gwp_id=lambda x: x["gwp_id"].fillna(""),
                    gwp=lambda x: x["gwp"].fillna(""),
                )
                .loc[
                    :,
                    [
                        "actor_id",
                        "year",
                        "sector_id",
                        "gas",
                        "gwp",
                        "gwp_id",
                        "emissions_tonnes",
                        "units",
                        "emissions_tonnes_co2e",
                        "assessment_report",
                        "datasource_id",
                    ],
                ]
            )

            list_of_dfs.append(df_sheet)

    # concatenate all the dataframes
    df = pd.concat(list_of_dfs)

    # ------------------------------------------
    # emissions
    # ------------------------------------------
    emissions_validated = [
        Emissions(
            id=f"{row.actor_id}:{row.year}:{row.gas}:{row.sector_id}",
            actor_id=row.actor_id,
            gas=GasType(row.gas),
            sector_id=row.sector_id,
            year=row.year,
            emissions=row.emissions_tonnes,
            units=row.units,
            datasource_id=datasource.id,
        ).model_dump()
        for row in df.query("gas.isin(@main_three)").itertuples()
    ]

    write_csv(
        output_dir=output_dir,
        name=Emissions.__tablename__,
        data=emissions_validated,
        mode="w",
    )

    # ------------------------------------------
    # emissions CO2eq
    # ------------------------------------------
    emissionsco2e_validated = [
        EmissionsCO2e(
            id=f"{row.actor_id}:{row.year}:{row.sector_id}:{row.gas}:{row.assessment_report}",
            actor_id=row.actor_id,
            sector_id=row.sector_id,
            gas=GasType(row.gas),
            gwp_id=row.gwp_id,
            assessment_report=AssessmentReport(row.assessment_report),
            year=row.year,
            emissions=row.emissions_tonnes_co2e,
            units="CO2 * tonne / yr",
            datasource_id=datasource.id,
        ).model_dump()
        for row in df.itertuples()
    ]

    write_csv(
        output_dir=output_dir,
        name=EmissionsCO2e.__tablename__,
        data=emissionsco2e_validated,
        mode="w",
    )

    gases_included = ", ".join(df["gas"].unique())
    df_total_sector = (
        df.groupby(["actor_id", "year", "sector_id", "assessment_report"])
        .sum("emissions_tonnes_co2e")
        .reset_index()
    )

    # ------------------------------------------
    # emissions total sector
    # ------------------------------------------
    emissionstotalsector_validated = [
        EmissionsTotalSector(
            id=f"{row.actor_id}:{row.year}:{row.sector_id}:{row.assessment_report}",
            actor_id=row.actor_id,
            sector_id=row.sector_id,
            year=row.year,
            emissions=row.emissions_tonnes_co2e,
            gases_included=gases_included,
            assessment_report=AssessmentReport(row.assessment_report),
            units="CO2 * tonne / yr",
            datasource_id=datasource.id,
        ).model_dump()
        for row in df_total_sector.itertuples()
    ]

    write_csv(
        output_dir=output_dir,
        name=EmissionsTotalSector.__tablename__,
        data=emissionstotalsector_validated,
        mode="w",
    )

    df_total_all_sector = (
        df_total_sector.query(
            "sector_id.isin(['crt:1','crt:2','crt:3','crt:4','crt:5','crt:6'])"
        )  # all sectors
        .groupby(["actor_id", "year", "assessment_report"])
        .sum("emissions_tonnes_co2e")
        .reset_index()
        .assign(aggregation_type="total")
    )

    df_total_ex_lulucf = (
        df_total_sector.query(
            "sector_id.isin(['crt:1','crt:2','crt:3','crt:5','crt:6'])"
        )  # excclude lulucf
        .groupby(["actor_id", "year", "assessment_report"])
        .sum("emissions_tonnes_co2e")
        .reset_index()
        .assign(aggregation_type="total_ex_lulucf")
    )

    df_emission_totals = pd.concat([df_total_all_sector, df_total_ex_lulucf]).drop(
        columns=["emissions_tonnes"]
    )

    # ------------------------------------------
    # emission total
    # sum across all sector and gases
    # total and excluding lulucf
    # ------------------------------------------
    # total across sector and gas
    emissionstotalco2e_validated = [
        EmissionsTotalCO2e(
            id=f"{row.actor_id}:{row.year}:{row.assessment_report}:{row.aggregation_type}",
            actor_id=row.actor_id,
            year=row.year,
            emissions=row.emissions_tonnes_co2e,
            aggregation_type=AggregationType(row.aggregation_type),
            gases_included=gases_included,
            units="CO2 * tonne / yr",
            assessment_report=AssessmentReport(row.assessment_report),
            datasource_id=datasource.id,
        ).model_dump()
        for row in df_emission_totals.itertuples()
    ]

    write_csv(
        output_dir=output_dir,
        name=EmissionsTotalCO2e.__tablename__,
        data=emissionstotalco2e_validated,
        mode="w",
    )


if __name__ == "__main__":
    main()


In [ ]:


    .drop(columns=["Base year"])
    .melt(id_vars="GHG emissions, kt CO2 equivalent", var_name='year', value_name='emissions_kt_co2_eq')
    .rename(columns={"GHG emissions, kt CO2 equivalent":"sector"})
    .loc[
        lambda d: pd.to_numeric(d["emissions_kt_co2_eq"], errors="coerce").notna()
    ]
    .assign(
        code=lambda d: d["sector"].str.split(" ", n=1).str[0],
        name=lambda d: d["sector"].str.split(" ", n=1).str[1],
        emissions=lambda d: d["emissions_kt_co2_eq"] * kt_to_tonne,
        units = "CO2 * tonne / yr"
    )
    .assign(
        code = lambda d: d['code'].str.rstrip('.'),
        actor_id = actor_id
    )
    .merge(df_codes, on=["code"])
    .loc[:, ["id_sector", "actor_id", "year", "emissions", "units"]]